# Prueba 3 - Faster R-CNN ResNet50-FPN para xView Detection

Este notebook cambia de un detector de una etapa a un detector de dos etapas para estudiar si Faster R-CNN se adapta mejor a escenas aereas densas y a objetos de tamano reducido. La prueba utiliza PyTorch y Torchvision con una arquitectura ResNet50-FPN preentrenada en COCO, sustituyendo la cabeza de clasificacion para las cuatro clases de xView y anadiendo la clase de fondo requerida por Faster R-CNN.

El flujo incluye la carga del dataset, el recuento de objetos por clase, la division en 6845 imagenes de entrenamiento y 761 de validacion, la definicion de un `Dataset` propio para xView, la construccion del modelo y una rutina completa de entrenamiento. Se entrena hasta 40 epocas con AdamW, programacion coseno del learning rate, checkpoint del mejor modelo y parada temprana. La validacion se calcula con IoU 0.50, AP por clase y mAP para comparar con las pruebas YOLO.

El mejor checkpoint aparece en la epoca 14 con `mAP@0.5 = 0.4618`. El desglose muestra `AP Small car = 0.7540`, `AP Bus = 0.3415`, `AP Truck = 0.1659` y `AP Building = 0.5857`, una mejora notable frente a YOLOv8 XS aunque con margen en las clases minoritarias. La ultima parte carga el mejor checkpoint, ejecuta inferencia sobre 852 imagenes de test y genera `prediction_fasterrcnn.json` junto con `submission_fasterrcnn.zip`.


In [2]:
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator

RUN_START = time.perf_counter()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORKDIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('device:', DEVICE)
print('WORKDIR:', WORKDIR)


torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
device: cuda
WORKDIR: /kaggle/working


In [3]:
categories = {0: 'Small car', 1: 'Bus', 2: 'Truck', 3: 'Building'}
num_classes = len(categories) + 1  # + background class for Faster R-CNN


def find_xview_root():
    kaggle_input = Path('/kaggle/input')
    candidates = [
        Path.cwd() / 'xview_detection',
        Path.cwd().parent / 'PROJECT' / 'xview_detection',
        Path('/kaggle/input/xview-detection/xview_detection'),
        Path('/kaggle/input/xview-detection'),
        Path('/kaggle/input/xview_detection'),
    ]
    for candidate in candidates:
        if (candidate / 'xview_det_train.json').exists():
            return candidate
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob('xview_det_train.json'))
        if matches:
            return matches[0].parent
    raise FileNotFoundError('No se encontro xview_det_train.json. Anade el dataset xView Detection como input.')

DATA_ROOT = find_xview_root()
JSON_FILE = DATA_ROOT / 'xview_det_train.json'
TRAIN_DIR = DATA_ROOT / 'xview_train'
TEST_DIR = DATA_ROOT / 'xview_test'

print('DATA_ROOT:', DATA_ROOT)
print('JSON_FILE:', JSON_FILE)
print('TRAIN_DIR exists:', TRAIN_DIR.exists())
print('TEST_DIR exists:', TEST_DIR.exists())


def load_geoimage(filename):
    path = DATA_ROOT / filename
    if not path.exists():
        raise FileNotFoundError(f'No se encontro la imagen: {path}')
    with rasterio.open(path, 'r') as src:
        image = np.zeros((src.height, src.width, src.count), dtype=src.profile['dtype'])
        for band in range(src.count):
            image[:, :, band] = src.read(band + 1)
    return image


DATA_ROOT: /kaggle/input/datasets/victordmv/visu-detect
JSON_FILE: /kaggle/input/datasets/victordmv/visu-detect/xview_det_train.json
TRAIN_DIR exists: True
TEST_DIR exists: True


In [4]:
with open(JSON_FILE) as f:
    json_data = json.load(f)

anns = []
counts = dict.fromkeys(categories.values(), 0)
annotations_by_image = {}
for ann in json_data['annotations'].values():
    annotations_by_image.setdefault(ann['image_id'], []).append(ann)

for img in json_data['images'].values():
    objects = []
    for ann in annotations_by_image.get(img['image_id'], []):
        cls_name = ann['category_id']
        counts[cls_name] += 1
        x1, y1, x2, y2 = ann['bbox']
        objects.append({
            'bbox': [float(x1), float(y1), float(x2), float(y2)],
            'category_id': cls_name,
        })
    anns.append({
        'filename': img['filename'],
        'image_id': img['image_id'],
        'width': img['width'],
        'height': img['height'],
        'objects': objects,
    })

anns_train, anns_valid = train_test_split(anns, test_size=0.1, random_state=SEED, shuffle=True)
print('counts:', counts)
print('Number of training images:', len(anns_train))
print('Number of validation images:', len(anns_valid))


counts: {'Small car': 188300, 'Bus': 6269, 'Truck': 10600, 'Building': 275943}
Number of training images: 6845
Number of validation images: 761


In [5]:
class XViewDetectionDataset(Dataset):
    def __init__(self, annotations, training=True):
        self.annotations = annotations
        self.training = training

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        image_np = load_geoimage(ann['filename'])
        image = torch.as_tensor(image_np, dtype=torch.float32).permute(2, 0, 1) / 255.0

        boxes = []
        labels = []
        for obj in ann['objects']:
            x1, y1, x2, y2 = obj['bbox']
            x1 = max(0.0, min(float(x1), ann['width'] - 1))
            y1 = max(0.0, min(float(y1), ann['height'] - 1))
            x2 = max(0.0, min(float(x2), ann['width'] - 1))
            y2 = max(0.0, min(float(y2), ann['height'] - 1))
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(list(categories.values()).index(obj['category_id']) + 1)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) if len(boxes) else torch.zeros((0,), dtype=torch.float32)
        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx], dtype=torch.int64),
            'area': area,
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image, target, ann['filename']


def collate_fn(batch):
    images, targets, filenames = zip(*batch)
    return list(images), list(targets), list(filenames)

BATCH_SIZE = 1
NUM_WORKERS = 2
train_ds = XViewDetectionDataset(anns_train, training=True)
valid_ds = XViewDetectionDataset(anns_valid, training=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn, pin_memory=True)

print('train batches:', len(train_loader))
print('valid batches:', len(valid_loader))


train batches: 6845
valid batches: 761


In [6]:
def build_fasterrcnn_model(num_classes):
    try:
        from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
        weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
        model = fasterrcnn_resnet50_fpn(weights=weights)
        print('Loaded COCO-pretrained Faster R-CNN weights')
    except Exception as exc:
        print('Could not load default torchvision weights:', exc)
        try:
            model = fasterrcnn_resnet50_fpn(pretrained=True)
            print('Loaded legacy pretrained=True weights')
        except Exception as exc2:
            print('Could not load legacy pretrained weights. Starting from random weights:', exc2)
            try:
                model = fasterrcnn_resnet50_fpn(weights=None, weights_backbone=None)
            except TypeError:
                model = fasterrcnn_resnet50_fpn(pretrained=False, pretrained_backbone=False)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Smaller anchors for satellite objects. Keep 3 aspect ratios so the pretrained RPN head remains shape-compatible.
    anchor_generator = AnchorGenerator(
        sizes=((4,), (8,), (16,), (32,), (64,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5,
    )
    model.rpn.anchor_generator = anchor_generator

    # xView images can contain hundreds of objects; the torchvision default of 100 detections is too restrictive.
    model.roi_heads.detections_per_img = 2000
    model.roi_heads.score_thresh = 0.05
    model.roi_heads.nms_thresh = 0.5
    if hasattr(model.rpn, '_pre_nms_top_n'):
        model.rpn._pre_nms_top_n = {'training': 4000, 'testing': 4000}
    if hasattr(model.rpn, '_post_nms_top_n'):
        model.rpn._post_nms_top_n = {'training': 2000, 'testing': 2000}
    return model

model = build_fasterrcnn_model(num_classes).to(DEVICE)
print(model.roi_heads.box_predictor)
print('detections_per_img:', model.roi_heads.detections_per_img)


Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 160MB/s]  


Loaded COCO-pretrained Faster R-CNN weights
FastRCNNPredictor(
  (cls_score): Linear(in_features=1024, out_features=5, bias=True)
  (bbox_pred): Linear(in_features=1024, out_features=20, bias=True)
)
detections_per_img: 2000


In [7]:
def area_intersection(boxes, box):
    xmin = np.maximum(np.min(boxes[:, 0::2], axis=1), np.min(box[0::2]))
    ymin = np.maximum(np.min(boxes[:, 1::2], axis=1), np.min(box[1::2]))
    xmax = np.minimum(np.max(boxes[:, 0::2], axis=1), np.max(box[0::2]))
    ymax = np.minimum(np.max(boxes[:, 1::2], axis=1), np.max(box[1::2]))
    w = np.maximum(xmax - xmin + 1.0, 0.0)
    h = np.maximum(ymax - ymin + 1.0, 0.0)
    return w * h


def area_union(boxes, box):
    area_ann = (np.max(box[0::2]) - np.min(box[0::2]) + 1.0) * (np.max(box[1::2]) - np.min(box[1::2]) + 1.0)
    area_pred = (np.max(boxes[:, 0::2], axis=1) - np.min(boxes[:, 0::2], axis=1) + 1.0) * (np.max(boxes[:, 1::2], axis=1) - np.min(boxes[:, 1::2], axis=1) + 1.0)
    return area_ann + area_pred - area_intersection(boxes, box)


def calc_iou(boxes, box):
    iou = area_intersection(boxes, box) / np.maximum(area_union(boxes, box), np.finfo(np.float64).eps)
    max_value = np.max(iou)
    max_index = np.argmax(iou)
    return max_value, max_index


def calc_ap(rec, prec):
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])


def compute_detection_metrics(annotations, predictions, iou_threshold=0.5):
    default_cls = 'BACKGROUND'
    y_true, y_pred = [], []
    tps, confidences = {cls: [] for cls in categories.values()}, {cls: [] for cls in categories.values()}
    for cls in categories.values():
        for filename in annotations:
            pred_boxes, pred_confidences = [], []
            if cls in predictions.get(filename, {}):
                pred_boxes = predictions[filename][cls]['bbox']
                pred_confidences = predictions[filename][cls]['confidence']
                order = np.argsort(-np.array(pred_confidences))
                pred_boxes = np.array(pred_boxes, dtype=float)[order]
                pred_confidences = np.array(pred_confidences, dtype=float)[order]
            else:
                pred_boxes = np.empty((0, 4), dtype=float)
                pred_confidences = np.array([], dtype=float)

            anno_boxes = np.array(annotations[filename].get(cls, {'bbox': []})['bbox'], dtype=float)
            anno_indices = list(range(len(anno_boxes)))

            for pred_idx, pred_box in enumerate(pred_boxes):
                iou_value, ann_index = calc_iou(anno_boxes, pred_box) if len(anno_boxes) > 0 else (-1, -1)
                if iou_value > iou_threshold and ann_index in anno_indices:
                    anno_indices.remove(int(ann_index))
                    tps[cls].append(1.0)
                    y_true.append(cls)
                else:
                    tps[cls].append(0.0)
                    y_true.append(default_cls)
                y_pred.append(cls)
                confidences[cls].append(float(pred_confidences[pred_idx]))

            y_true.extend([cls] * len(anno_indices))
            y_pred.extend([default_cls] * len(anno_indices))

    ap_by_class = {}
    for cls in categories.values():
        if len(confidences[cls]) == 0:
            ap_by_class[cls] = 0.0
            continue
        order = np.argsort(-np.array(confidences[cls]))
        tp = np.cumsum(np.array(tps[cls])[order], dtype=float)
        recall = tp / np.maximum(np.sum(np.array(y_true) == cls), np.finfo(np.float64).eps)
        precision = tp / np.maximum(np.arange(1, np.sum(np.array(y_pred) == cls) + 1), np.finfo(np.float64).eps)
        ap_by_class[cls] = float(calc_ap(recall, precision))
    return float(np.mean(list(ap_by_class.values()))), ap_by_class


@torch.no_grad()
def predict_loader(model, loader, score_threshold=0.05):
    model.eval()
    annotations, predictions = {}, {}
    for images, targets, filenames in tqdm(loader, desc='Predict'):
        images = [img.to(DEVICE) for img in images]
        outputs = model(images)
        for target, output, filename in zip(targets, outputs, filenames):
            annotations.setdefault(filename, {})
            predictions.setdefault(filename, {})

            for box, label in zip(target['boxes'].cpu().numpy(), target['labels'].cpu().numpy()):
                cls = categories[int(label) - 1]
                annotations[filename].setdefault(cls, {'bbox': []})
                annotations[filename][cls]['bbox'].append(box.tolist())

            boxes = output['boxes'].detach().cpu().numpy()
            labels = output['labels'].detach().cpu().numpy()
            scores = output['scores'].detach().cpu().numpy()
            keep = scores >= score_threshold
            for box, label, score in zip(boxes[keep], labels[keep], scores[keep]):
                if int(label) <= 0 or int(label) > len(categories):
                    continue
                cls = categories[int(label) - 1]
                predictions[filename].setdefault(cls, {'bbox': [], 'confidence': []})
                predictions[filename][cls]['bbox'].append(box.tolist())
                predictions[filename][cls]['confidence'].append(float(score))
    return annotations, predictions


In [8]:
EPOCHS = 40
PATIENCE = 8
BEST_CKPT_PATH = WORKDIR / 'fasterrcnn_best.pth'
LAST_CKPT_PATH = WORKDIR / 'fasterrcnn_last.pth'
TRAINING_LOG_PATH = WORKDIR / 'fasterrcnn_training_log.jsonl'

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_map = -1.0
bad_epochs = 0

if BEST_CKPT_PATH.exists():
    checkpoint = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    best_map = float(checkpoint.get('best_map', -1.0))
    print('Loaded previous best checkpoint:', BEST_CKPT_PATH, 'best_map=', best_map)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    start = time.perf_counter()

    for images, targets, filenames in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}'):
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in target.items()} for target in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_value for loss_value in loss_dict.values())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        epoch_loss += float(loss.detach().cpu())

    scheduler.step()
    train_loss = epoch_loss / max(len(train_loader), 1)
    annotations, predictions = predict_loader(model, valid_loader, score_threshold=0.05)
    val_map, ap_by_class = compute_detection_metrics(annotations, predictions, iou_threshold=0.5)
    elapsed = time.perf_counter() - start

    log_row = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_map_50': val_map,
        'ap_by_class': ap_by_class,
        'lr': optimizer.param_groups[0]['lr'],
        'elapsed_sec': elapsed,
    }
    with open(TRAINING_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(log_row) + '\n')

    print(f'Epoch {epoch + 1}: loss={train_loss:.4f} mAP@0.5={val_map:.4f} lr={optimizer.param_groups[0]["lr"]:.2e} time={elapsed:.1f}s')
    print('AP by class:', ap_by_class)

    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_map': best_map,
    }, LAST_CKPT_PATH)

    if val_map > best_map:
        best_map = val_map
        bad_epochs = 0
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_map': best_map,
            'ap_by_class': ap_by_class,
        }, BEST_CKPT_PATH)
        print('Saved new best checkpoint:', BEST_CKPT_PATH)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print('Early stopping: no mAP improvement for', PATIENCE, 'epochs')
            break

print('Best mAP@0.5:', best_map)
print('Best checkpoint:', BEST_CKPT_PATH)


Predict: 100%|██████████| 761/761 [01:25<00:00,  8.85it/s]


Epoch 1: loss=0.8828 mAP@0.5=0.3464 lr=9.98e-05 time=1666.7s
AP by class: {'Small car': 0.6598827307718618, 'Bus': 0.18715911218494602, 'Truck': 0.0813650946230471, 'Building': 0.4573511540103814}
Saved new best checkpoint: /kaggle/working/fasterrcnn_best.pth


Predict: 100%|██████████| 761/761 [01:26<00:00,  8.84it/s]


Epoch 2: loss=0.8112 mAP@0.5=0.4022 lr=9.94e-05 time=1661.1s
AP by class: {'Small car': 0.7313239166257366, 'Bus': 0.22211958263513937, 'Truck': 0.12398319245477796, 'Building': 0.5313636691126862}
Saved new best checkpoint: /kaggle/working/fasterrcnn_best.pth


Predict: 100%|██████████| 761/761 [01:25<00:00,  8.94it/s]


...

Epoch 20: loss=0.4470 mAP@0.5=0.4236 lr=5.05e-05 time=1645.7s
AP by class: {'Small car': 0.7253264910753884, 'Bus': 0.2658222577766074, 'Truck': 0.13174162960731478, 'Building': 0.571346764495567}


Predict: 100%|██████████| 761/761 [01:25<00:00,  8.90it/s]


Epoch 21: loss=0.4295 mAP@0.5=0.4289 lr=4.66e-05 time=1646.1s
AP by class: {'Small car': 0.7324897141982043, 'Bus': 0.28083747555949257, 'Truck': 0.1311982594529148, 'Building': 0.5711160825280555}


Predict: 100%|██████████| 761/761 [01:25<00:00,  8.88it/s]


Epoch 22: loss=0.4099 mAP@0.5=0.4163 lr=4.28e-05 time=1646.4s
AP by class: {'Small car': 0.7234987752496831, 'Bus': 0.2568360546823232, 'Truck': 0.12233951571019004, 'Building': 0.5623285299559027}
Early stopping: no mAP improvement for 8 epochs
Best mAP@0.5: 0.4617620754975037
Best checkpoint: /kaggle/working/fasterrcnn_best.pth


In [9]:
if not BEST_CKPT_PATH.exists():
    raise FileNotFoundError('No se encontro fasterrcnn_best.pth. Ejecuta primero la celda de entrenamiento.')

checkpoint = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print('Loaded best checkpoint from epoch:', checkpoint.get('epoch'))
print('Best validation mAP@0.5:', checkpoint.get('best_map'))

annotations, predictions = predict_loader(model, valid_loader, score_threshold=0.05)
val_map, ap_by_class = compute_detection_metrics(annotations, predictions, iou_threshold=0.5)
print('Validation mAP@0.5:', val_map)
print('AP by class:', ap_by_class)


Loaded best checkpoint from epoch: 14
Best validation mAP@0.5: 0.4617620754975037


Predict: 100%|██████████| 761/761 [01:26<00:00,  8.83it/s]


Validation mAP@0.5: 0.4617620754975037
AP by class: {'Small car': 0.7539857168727293, 'Bus': 0.34146973471291736, 'Truck': 0.16592875272515106, 'Building': 0.5856640976792171}


In [10]:
class XViewTestDataset(Dataset):
    def __init__(self, test_dir):
        self.paths = sorted(Path(test_dir).rglob('*.tif'))
        if not self.paths:
            raise FileNotFoundError(f'No se encontraron .tif en {test_dir}')

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        filename = path.relative_to(DATA_ROOT).as_posix()
        image_np = load_geoimage(filename)
        image = torch.as_tensor(image_np, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return image, filename


def collate_test_fn(batch):
    images, filenames = zip(*batch)
    return list(images), list(filenames)

SCORE_THRESHOLD_TEST = 0.05
test_ds = XViewTestDataset(TEST_DIR)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_test_fn, pin_memory=True)
print('Number of testing images:', len(test_ds))

checkpoint = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

predictions_data = {'images': {}, 'annotations': {}, 'categories': categories}
ann_idx = 0

with torch.no_grad():
    for img_idx, (images, filenames) in enumerate(tqdm(test_loader, desc='Test inference')):
        images = [img.to(DEVICE) for img in images]
        outputs = model(images)
        for filename, output in zip(filenames, outputs):
            boxes = output['boxes'].detach().cpu().numpy()
            labels = output['labels'].detach().cpu().numpy()
            scores = output['scores'].detach().cpu().numpy()
            keep = scores >= SCORE_THRESHOLD_TEST

            predictions_data['images'][img_idx] = {
                'image_id': filename.split('/')[-1],
                'filename': filename,
                'num_objects': int(np.sum(keep)),
                'width': 640,
                'height': 640,
            }

            for box, label, score in zip(boxes[keep], labels[keep], scores[keep]):
                if int(label) <= 0 or int(label) > len(categories):
                    continue
                x1, y1, x2, y2 = box.tolist()
                predictions_data['annotations'][ann_idx] = {
                    'image_id': filename.split('/')[-1],
                    'category_id': categories[int(label) - 1],
                    'bbox': [int(x1), int(y1), int(x2), int(y2)],
                    'confidence': float(score),
                }
                ann_idx += 1

prediction_path = WORKDIR / 'prediction_fasterrcnn.json'
submission_path = WORKDIR / 'submission_fasterrcnn.zip'
with open(prediction_path, 'w', encoding='utf-8') as outfile:
    json.dump(predictions_data, outfile)

with zipfile.ZipFile(submission_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname='prediction.json')

print('Archivos generados:')
print(prediction_path, '-', prediction_path.stat().st_size, 'bytes')
print(submission_path, '-', submission_path.stat().st_size, 'bytes')

try:
    from IPython.display import FileLink, display
    display(FileLink(str(submission_path)))
except Exception:
    pass


Number of testing images: 852


Test inference: 100%|██████████| 852/852 [01:37<00:00,  8.76it/s]


Archivos generados:
/kaggle/working/prediction_fasterrcnn.json - 21200556 bytes
/kaggle/working/submission_fasterrcnn.zip - 3179330 bytes


/kaggle/working/submission_fasterrcnn.zip

In [11]:
TOTAL_TIME = time.perf_counter() - RUN_START
print(f'Tiempo total: {TOTAL_TIME / 60:.2f} min')
print('Output principal:', WORKDIR / 'submission_fasterrcnn.zip')
print('Checkpoint principal:', WORKDIR / 'fasterrcnn_best.pth')


Tiempo total: 609.32 min
Output principal: /kaggle/working/submission_fasterrcnn.zip
Checkpoint principal: /kaggle/working/fasterrcnn_best.pth
